<a href="https://colab.research.google.com/github/aditisankara/lucidrt/blob/feat%2Fdata-pipeline/src/notebooks/lucidrt_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q "transformers>=4.44" "datasets>=2.20"

In [3]:
from datasets import load_dataset

raw = load_dataset("lighteval/med_paragraph_simplification")

print("splits + sizes:", {k: len(v) for k, v in raw.items()})
first = next(iter(raw.values()))
print("columns:", first.column_names)
print("\none example:")
print(first[0])

README.md:   0%|          | 0.00/526 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.48MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  744kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3568 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/411 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/480 [00:00<?, ? examples/s]

splits + sizes: {'train': 3568, 'validation': 411, 'test': 480}
columns: ['query', 'answer']

one example:
{'query': 'Two trials met the inclusion criteria. One compared 2% ketanserin ointment in polyethylene glycol (PEG) with PEG alone, used twice a day by 40 participants with arterial leg ulcers, for eight weeks or until healing, whichever was sooner. One compared topical application of blood-derived concentrated growth factor (CGF) with standard dressing (polyurethane film or foam); both applied weekly for six weeks by 61 participants with non-healing ulcers (venous, diabetic arterial, neuropathic, traumatic, or vasculitic). Both trials were small, reported results inadequately, and were of low methodological quality. Short follow-up times (six and eight weeks) meant it would be difficult to capture sufficient healing events to allow us to make comparisons between treatments. One trial demonstrated accelerated wound healing in the ketanserin group compared with the control group. In

In [4]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("vocab size:", tok.vocab_size, "| pad:", tok.pad_token, "| eos:", tok.eos_token)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

vocab size: 151643 | pad: <|endoftext|> | eos: <|im_end|>


In [5]:
PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.\n\n"
    "Medical text:\n{technical}\n\nPlain language:\n"
)

ex = raw["train"][0]
prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
full_text = prompt + ex["answer"] + tok.eos_token   # what the model will learn from: instruction+input, then the answer

enc = tok(full_text)

print("num tokens:", len(enc["input_ids"]))
print("first 20 ids:", enc["input_ids"][:20])
print("round-trip decode:", tok.decode(enc["input_ids"][:20]))

num tokens: 878
first 20 ids: [50, 70206, 279, 2701, 6457, 1467, 1119, 14396, 4128, 264, 8720, 646, 3535, 382, 52554, 1467, 510, 11613, 19080, 2270]
round-trip decode: Simplify the following medical text into plain language a patient can understand.

Medical text:
Two trials met


In [6]:
import torch
from torch.utils.data import Dataset

MAX_LEN = 512

class MedSimplifyDataset(Dataset):
    def __init__(self, split):
        self.data = split          # a HF split, e.g. raw["train"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # TODO(you): same idea as Cell 4, but:
        #   1. ex = self.data[idx]; build prompt + full_text (as in Cell 4)
        #   2. tokenize WITH truncation=True, max_length=MAX_LEN   <- so the 878-token ones fit
        #   3. return a dict of 1-D tensors:
        #        {"input_ids": <tensor>, "attention_mask": <tensor>}
        #      The tokenizer gives python LISTS; wrap each with torch.tensor(..., dtype=torch.long)
        ex = self.data[idx]
        prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
        full_text = prompt + ex["answer"] + tok.eos_token
        enc = tok(full_text, truncation=True, max_length=MAX_LEN)
        return {
            "input_ids" : torch.tensor(enc["input_ids"]),
            "attention_mask" : torch.tensor(enc["attention_mask"])
        }

# quick self-test:
ds = MedSimplifyDataset(raw["train"])
print("dataset size:", len(ds))
item = ds[0]
print("keys:", item.keys())
print("input_ids:", item["input_ids"].shape, item["input_ids"].dtype)

dataset size: 3568
keys: dict_keys(['input_ids', 'attention_mask'])
input_ids: torch.Size([512]) torch.int64


In [7]:
from torch.utils.data import DataLoader

ds = MedSimplifyDataset(raw["train"])
bad_loader = DataLoader(ds, batch_size=4)   # no collate_fn yet

try:
    batch = next(iter(bad_loader))
    print("input_ids:", batch["input_ids"].shape)
except Exception as e:
    print("ERROR:", type(e).__name__, "-", str(e)[:160])

ERROR: RuntimeError - stack expects each tensor to be equal size, but got [512] at entry 0 and [456] at entry 1


In [10]:
from dataclasses import dataclass

@dataclass
class PadCollator:
    pad_token_id: int

    def __call__(self, batch):
        # batch = list of dicts, each {"input_ids":[L_i], "attention_mask":[L_i]} with DIFFERENT L_i
        max_len = max(example["input_ids"].size(0) for example in batch)

        input_ids, masks = [], []
        for ex in batch:
            ids, mask = ex["input_ids"], ex["attention_mask"]
            pad_len = max_len - ids.size(0)          # how many pad slots this example needs

            # TODO: right-pad `ids` with self.pad_token_id, and `mask` with 0, each by pad_len.
            #   hint: torch.cat([ids,  torch.full((pad_len,), self.pad_token_id, dtype=torch.long)])
            #         torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])
            ids  = torch.cat([ids, torch.full((pad_len, ), self.pad_token_id, dtype=torch.long)])
            mask = torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])

            input_ids.append(ids)
            masks.append(mask)

        # TODO: stack the lists of 1-D tensors into 2-D [B, max_len] tensors (hint: torch.stack)
        return {"input_ids": torch.stack(input_ids), "attention_mask": torch.stack(masks)}


# test it with a real DataLoader:
collator = PadCollator(pad_token_id=tok.pad_token_id)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collator)
batch = next(iter(loader))
print("input_ids:", batch["input_ids"].shape)       # expect [4, max_len_in_batch]
print("attention_mask:", batch["attention_mask"].shape)
print("mask row 0 (1=real, 0=pad):", batch["attention_mask"][0].tolist()[-10:])

input_ids: torch.Size([4, 512])
attention_mask: torch.Size([4, 512])
mask row 0 (1=real, 0=pad): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [12]:
def make_loaders(batch_size=4):
    collator = PadCollator(pad_token_id=tok.pad_token_id)
    train_dl = DataLoader(MedSimplifyDataset(raw["train"]),
                          batch_size=batch_size, shuffle=True,  collate_fn=collator)
    val_dl   = DataLoader(MedSimplifyDataset(raw["validation"]),
                          batch_size=batch_size, shuffle=False, collate_fn=collator)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()
print("train batches:", len(train_dl), "| val batches:", len(val_dl))
xb = next(iter(train_dl))
print("one batch:", xb["input_ids"].shape)

train batches: 892 | val batches: 103
one batch: torch.Size([4, 512])
